![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 42 -- Case Study: Lost in Hyperspace (Solution)

The task of this morning's case study, as the contestants of the **IOAI 2024 (Burgas) On-Site Round** received it
(the ML task). The same notebook with the author's solution at the end: 6 symmetric copies of every training cube, PCA to 299 components, and one feature from the at-home task.

**Source and licence.** Adapted from the notebooks `On-Site-Round/Lost_in_Hyperspace/Lost_in_Hyperspace.ipynb` and
`Solution/Lost_in_Hyperspace_Solution.ipynb` of [IOAI-official/IOAI-2024](https://github.com/IOAI-official/IOAI-2024)
by the organisers of IOAI 2024, licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes: this
cell, the setup cell below and the last line are added; the pictures are loaded from the repository; a cell that explains the author's solution is added before it. Nothing
else is changed.

**How to run it.** In Colab, choose *Run all*. The setup cell downloads the task's files (about 30 MB) from the
organisers' repository into the folders the notebook expects. Everything runs on a CPU in under a minute.


In [ ]:
# Course setup (not part of the task): the task's files, in the folders the notebook expects.
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/IOAI-official/IOAI-2024/main/On-Site-Round/Lost_in_Hyperspace/"
FOLDER = os.path.join(os.getcwd(), "Lost_in_Hyperspace")
FILES = ["training_set/ml_data_onsite_start.pickle", "Solution/test_set/ml_data_onsite_final_test.pickle"]
for name in FILES:
    target = os.path.join(FOLDER, name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        urllib.request.urlretrieve(BASE + name, target)
os.chdir(os.path.join(FOLDER, "Solution"))
print(os.getcwd())

<img src="https://raw.githubusercontent.com/IOAI-official/IOAI-2024/main/On-Site-Round/Lost_in_Hyperspace/Solution/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgaria), On-Site Round](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2024/blob/main/On-Site-Round/Lost_in_Hyperspace/Solutioin/Lost_in_Hyperspace_Solution.ipynb)

# Lost in a Hyperspace: Baseline Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

SCALING_WEIGHTS = [100/15, 100/8, 100/100]

In [ ]:
data = pd.read_pickle('../training_set/ml_data_onsite_start.pickle')
for key in data.keys():
  print(key)

In [ ]:
for key in data['X'].keys():
  print(key)

In [ ]:
for key in data['y'].keys():
  print(key)

In [ ]:
X_train = data['X']['train']
y_train = data['y']['train']

X_val = data['X']['val']
y_val = data['y']['val']

X_test = data['X']['live_test']

In [ ]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape

In [ ]:
def vis(arr):
  plt.figure(figsize=(8, 8))

  cnt = 1
  for z in range(5):
    for q in range(6):
      plt.subplot(5, 6, cnt)
      plt.imshow(arr[:, :, z, q], vmin=-40, vmax=40, cmap='hsv')
      plt.grid()
      plt.axis('off')
      cnt += 1
  plt.tight_layout()

In [ ]:
vis(X_train[0])

## Functions for result evaluation / writing predictions

Do not change it!

In [ ]:
def test_solution(X_train, y_train, X_val, y_val, feature_num=0):
    assert X_train.shape[-1] <= 300, "Too many features! Should be less than 300"
    assert X_val.shape[-1] <= 300, "Too many features! Should be less than 300"

    model =  LinearRegression().fit(
        X_train,
        y_train[:, feature_num]
    )
    predictions = model.predict(X_val)
    rmse = mean_squared_error(
        predictions,
        y_val[:, feature_num]
    )**.5
    normalized_rmse = rmse * SCALING_WEIGHTS[feature_num]
    print(f"Property #{feature_num}:    raw RMSE={rmse:.6f}")
    print(f"Property #{feature_num}: scaled RMSE={normalized_rmse:.6f}")
    return round(normalized_rmse, 6)

## Let's try a baseline solution

In [ ]:
def dummy_feature_extractor(X):
    X_new = X.reshape((X.shape[0], -1)) # ravel
    X_new = X_new[:, :300] # pick first 300 features
    return X_new

In [ ]:
 dummy_feature_extractor(X_train).shape

In [ ]:
%%time
total_score = 0
for feature_number in range(3):
  total_score += test_solution(
      dummy_feature_extractor(X_train),
      y_train,
      dummy_feature_extractor(X_val),
      y_val,
      feature_num=feature_number
  )
  print()
total_score /= 3
print('='*16)
print(f"Total score = {total_score:.6f}")

# How to prepare the answer files

In [ ]:
def generate_predictions(X_train, y_train, X_test, feature_num=0):
    assert X_train.shape[-1] <= 300
    assert X_test.shape[-1] <= 300

    model =  LinearRegression().fit(
        X_train,
        y_train[:, feature_num]
    )
    predictions = model.predict(X_test)
    return predictions


## Generate solutions and write to the file
combined = {'ID': np.arange(X_test.shape[0])}

for feature_number in range(3):
    predictions = generate_predictions(
        dummy_feature_extractor(X_train),
        y_train,
        dummy_feature_extractor(X_test),
        feature_num=feature_number
    )

    combined[f'y{feature_number+1}'] = predictions

pd.DataFrame(combined).to_csv('predictions.csv', index=False)

In [ ]:
# load the test dataset
loaded = pd.read_pickle("./test_set/ml_data_onsite_final_test.pickle")
X_test_final = loaded['X']['final_test']


# make final predictions
combined = {'ID': np.arange(X_test_final.shape[0])}

for feature_number in range(3):
    predictions = generate_predictions(
        dummy_feature_extractor(X_train),
        y_train,
        dummy_feature_extractor(X_test_final),
        feature_num=feature_number
    )

    combined[f'y{feature_number+1}'] = predictions

pd.DataFrame(combined).to_csv('final_predictions.csv', index=False)

**How the author's solution works** (the steps of the Day 42 slides):

1. **Copies.** The first three axes of a cube can be swapped without changing its properties (the task's hint), so
   `symmetrize_x` adds the 5 other orders of the axes: 12,144 training cubes instead of 2,024,
   with the same answers.
2. **PCA.** `ravelize` turns each cube into 750 numbers; `PCA(n_components=299)` makes 299 new columns, each a
   weighted mix of all 750, ordered by spread. The table of cubes has rank 252, so 299 columns keep
   everything a linear model can use from the 750. Every number of a cube has a twin through its centre, so only
   268 of the 750 columns are different: keeping those
   (`np.unique(X_flat, axis=1, return_index=True)`) scores the same as PCA here.
3. **One feature from the at-home task.** `make_bg`: the smallest number of thread 3 minus the largest of thread 2.
   The author knew it from the earlier task. The data shows it too: a linear model on the smallest and the largest
   number of each thread puts +1.00 on the smallest of thread 3,
   -1.00 on the largest of thread 2, and almost 0 on the rest.
4. **The fixed model.** The given `LinearRegression`, scored by the given `test_solution` on the validation cubes.

Measured with the task's scorer on the 376 validation cubes (lower is better): the starter 6.82;
PCA alone 4.95 (the 268 different columns: 4.95);
with the copies 3.71; with the extra feature
1.65.

## Author Solution

- Replicate the data 5 more times, swapping axes (number of permutations for three axes)

- (Optionally) Delete duplicate columns

- Use PCA, it helps

- (Author's knowledge) One of the features is from the previous task, gives biggest improvement, so home-task knowledge definitely helps

Most successful teams combined some replication(=augmentation), features from home task, and PCA. Validation set should be a good indicator whether particular feature is a good idea to add.

In [ ]:
def symmetrize_x(X_tr, y_tr):
    xxx = [
        X_tr,
        X_tr.swapaxes(1, 2),
        X_tr.swapaxes(1, 3),
        X_tr.swapaxes(2, 3),
        X_tr.swapaxes(1, 3).swapaxes(1, 2),
        X_tr.swapaxes(1, 3).swapaxes(2, 3),
    ]
    return np.concatenate(xxx), np.vstack([y_tr]*6)

In [ ]:
def ravelize(X):
    return X.reshape((X.shape[0], -1))

In [ ]:
def make_bg(X):
    bgs = []
    for _x in X:
        bg = _x[:,:,:,3].ravel().min() - _x[:,:,:,2].ravel().max()
        bgs.append(bg)
    bgs = np.array(bgs)
    return bgs[:, None]

In [ ]:
X_train_symm, y_train_symm = symmetrize_x(X_train, y_train)

In [ ]:
X_train_bg = make_bg(X_train_symm)
X_val_bg = make_bg(X_val)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=299)
X_train_pca = pca.fit_transform(ravelize(X_train_symm))
X_val_pca = pca.transform(ravelize(X_val))

In [ ]:
%%time
total_score = 0
for feature_number in range(3):
  total_score += test_solution(
      X_train_pca,
      y_train_symm,
      X_val_pca,
      y_val,
      feature_num=feature_number
  )
  print()
total_score /= 3
print('='*16)
print(f"Total score = {total_score:.6f}")

In [ ]:
%%time
total_score = 0
# something close to the best possible solution
for feature_number in range(3):
  total_score += test_solution(
      np.concatenate([X_train_pca, X_train_bg], axis=-1),
      y_train_symm,
      np.concatenate([X_val_pca, X_val_bg], axis=-1),
      y_val,
      feature_num=feature_number
  )
  print()
total_score /= 3
print('='*16)
print(f"Total score = {total_score:.6f}")

Adapted By **Sattam Altwaim**, from the notebooks of the organisers of IOAI 2024 (CC BY 4.0).